In [ ]:
'''
Updated 01-11-2026

Create per-star folders from the HARPS V2 grouping (HARPS version of create_table.ipynb)
Same layout as UVES, so ASSET can read it:
    <dataset>/<star>/meta/group_df.pkl, meta/fits.npy
    <dataset>/<star>/spec/sK.npy, spec/sH.npy
    <dataset>/<star>/wavelength/wK.npy, wH.npy, rvK.npy, rvH.npy
'''
import json
import re
import sqlite3
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo(start=Path.cwd()):
    '''Walk up from the notebook until config/harps.json is found.'''
    for folder in [start, *start.parents]:
        if (folder / 'config' / 'harps.json').exists():
            return folder
    raise FileNotFoundError('config/harps.json not found above ' + str(start))


config = json.loads((find_repo() / 'config' / 'harps.json').read_text())
DATA_DIR = Path(config['data_dir'])
DB_PATH = DATA_DIR / 'state.sqlite'
GROUPS_PATH = DATA_DIR / 'grouping' / 'v2' / 'harps_full_metadata_v2.pkl'
OUT_DIR = DATA_DIR / 'dataset'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_SPECTRA = config['dataset']['min_spectra']
THREADS = config['dataset']['load_threads']
LINES_A = {'K': 3933.66, 'H': 3968.47}  # Angstrom, as GetData.ipynb

In [ ]:
### Load metadata ###
metadata = pd.read_pickle(GROUPS_PATH)
metadata = metadata[metadata['method'].isin(['name', 'cone'])]  # non-stellar + unresolved stay out of the search

db = sqlite3.connect(f'file:{DB_PATH.as_posix()}?mode=ro', uri=True)
header = pd.read_sql_query(
    "SELECT dp_id, json_extract(header_json, '$.OBJECT') AS Object, "
    "json_extract(header_json, '$.SPEC_RES') AS SPEC_RES, json_extract(header_json, '$.SNR') AS SNR "
    "FROM spectra WHERE status = 'done'",
    db,
)
db.close()
metadata = metadata.merge(header, on='dp_id', how='left', validate='one_to_one')

sizes = metadata.groupby('star_id')['dp_id'].transform('size')
print(f'Stellar: {len(metadata)} spectra, {metadata["star_id"].nunique()} stars')
metadata = metadata[sizes >= MIN_SPECTRA]
print(f'With >= {MIN_SPECTRA} spectra: {len(metadata)} spectra, {metadata["star_id"].nunique()} stars')
print('Missing header values:', metadata[['Object', 'SPEC_RES', 'SNR']].isna().sum().to_dict())

In [ ]:
### Wavelength grid ###
# Every cut-out should share one grid: taken from the first spectrum, checked on every spectrum below
first = np.load(DATA_DIR / metadata['file'].iloc[0])
wave = {line: first[f'wave_{line}'] * 10.0 for line in LINES_A}  # nm -> Angstrom (ASSET.X_corr works in Angstrom)
rv = {line: 3e5 * (wave[line] - centre) / centre for line, centre in LINES_A.items()}  # same formula as GetData
for line in LINES_A:
    print(f'{line}: {len(wave[line])} px, {wave[line][0]:.2f}-{wave[line][-1]:.2f} A, '
          f'rv {rv[line][0]:.1f} to {rv[line][-1]:.1f} km/s')

In [ ]:
### Build folders ###
def folder_name(star_id):
    '''star_id as a folder name: GaiaDR3 123 -> GaiaDR3_123, SIMBAD * alf Cen B -> SIMBAD_alf_Cen_B'''
    name = re.sub(r'[^A-Za-z0-9+.-]+', '_', star_id).strip('_')
    return name.rstrip('.')


def load_spectrum(path):
    '''Flux of one spectrum, or None if its grid differs from the common one.'''
    with np.load(path) as z:
        same_grid = all(len(z[f'wave_{line}']) == len(wave[line])
                        and np.allclose(z[f'wave_{line}'] * 10.0, wave[line], rtol=0, atol=1e-4)  # 1e-4 A = 1% of a pixel
                        for line in LINES_A)
        return (z['flux_K'], z['flux_H']) if same_grid else None


def build_star(star_id, group_df, pool):
    group_df = group_df.sort_values('t_min').reset_index(drop=True)
    group_path = OUT_DIR / folder_name(star_id)
    meta_path = group_path / 'meta' / 'group_df.pkl'

    # group_df is written last, so a folder that has it is complete
    if meta_path.exists():
        built = pd.read_pickle(meta_path)
        left_out = group_df.loc[~group_df['dp_id'].isin(built['dp_id']), 'dp_id'].tolist()
        return group_path.name, len(built), left_out, 0

    paths = [DATA_DIR / f for f in group_df['file']]
    loaded = list(pool.map(load_spectrum, paths))
    keep = np.array([spec is not None for spec in loaded])
    bad_grid = group_df.loc[~keep, 'dp_id'].tolist()
    group_df = group_df[keep].reset_index(drop=True)
    if len(group_df) == 0:  # every spectrum on the 2026 grid: nothing to stack
        return group_path.name, 0, bad_grid, 0

    group_sK = np.array([spec[0] for spec in loaded if spec is not None])
    group_sH = np.array([spec[1] for spec in loaded if spec is not None])
    not_finite = int((~np.isfinite(group_sK)).any(axis=1).sum() + (~np.isfinite(group_sH)).any(axis=1).sum())
    group_fits = np.array([str(p) for p, k in zip(paths, keep) if k])

    for sub in ('spec', 'wavelength', 'meta'):
        (group_path / sub).mkdir(parents=True, exist_ok=True)

    np.save(group_path / 'spec' / 'sK.npy', group_sK)
    np.save(group_path / 'spec' / 'sH.npy', group_sH)
    np.save(group_path / 'meta' / 'fits.npy', group_fits)
    for line in LINES_A:
        np.save(group_path / 'wavelength' / f'w{line}.npy', wave[line])
        np.save(group_path / 'wavelength' / f'rv{line}.npy', rv[line])
    group_df.to_pickle(meta_path)

    return group_path.name, len(group_df), bad_grid, not_finite

In [ ]:
groups = metadata.groupby('star_id', sort=False)
assert metadata.groupby(metadata['star_id'].map(folder_name))['star_id'].nunique().max() == 1, 'folder name clash'

index, all_bad_grid, all_not_finite = [], [], 0
with ThreadPoolExecutor(THREADS) as pool:
    for i, (star_id, group_df) in enumerate(groups):
        name, n, bad_grid, not_finite = build_star(star_id, group_df, pool)
        index.append({'star_id': star_id, 'folder': name, 'star_label': group_df['star_label'].iloc[0],
                      'gaia_dr3': group_df['gaia_dr3'].iloc[0], 'n_spectra': n})
        all_bad_grid += bad_grid
        all_not_finite += not_finite
        if (i + 1) % 500 == 0:
            print(f'{i + 1}/{groups.ngroups}', flush=True)

index = pd.DataFrame(index).query('n_spectra > 0').sort_values('n_spectra', ascending=False).reset_index(drop=True)
index.to_csv(OUT_DIR / 'dataset_index.csv', index=False)
pd.Series(all_bad_grid, name='dp_id').to_csv(OUT_DIR / 'bad_grid.csv', index=False)

print(f'Stars built: {len(index)}, spectra {index["n_spectra"].sum()}')
print(f'Spectra left out (different grid): {len(all_bad_grid)}')
print(f'Stacked spectra with non-finite flux: {all_not_finite}')
print(index.head(10).to_string())

In [ ]:
### Check the output ###
for name in [index['folder'].iloc[0], index['folder'].iloc[len(index) // 2], index['folder'].iloc[-1]]:
    star = OUT_DIR / name
    group_df = pd.read_pickle(star / 'meta' / 'group_df.pkl')
    sK, sH = np.load(star / 'spec' / 'sK.npy', mmap_mode='r'), np.load(star / 'spec' / 'sH.npy', mmap_mode='r')
    assert sK.shape == sH.shape == (len(group_df), len(wave['K']))
    with np.load(DATA_DIR / group_df['file'].iloc[-1]) as z:  # last row must be the last spectrum in the stack
        assert np.array_equal(sK[-1], z['flux_K']) and np.array_equal(sH[-1], z['flux_H'])
    assert group_df['t_min'].is_monotonic_increasing
    print(f'{name}: {sK.shape} ok')

size_gb = sum(f.stat().st_size for f in OUT_DIR.rglob('*.npy')) / 1e9
print(f'Dataset size: {size_gb:.1f} GB')